<a href="https://colab.research.google.com/github/masaki-kawa/uts-study-notes/blob/main/data/raw/colab/Deep_Learning_Lab6_Exercise3_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN with Tensorflow

# Generate Shakespeare Text

In this exercise, we will use a sequence model (LSTM) to analyse Shakespeare's work and generate new text.
The dataset can be dowloaded from here: https://www.kaggle.com/datasets/adarshpathak/shakespeare-text

## Objective

Our goal is to train an LSTM model with embeddings that will generate new text.

## Instructions

This is a guided exercise where some of the code have already been pre-defined. Your task is to fill the remaining part of the code (it will be highlighted with placehoders) to train and evaluate your model.

This exercise is split in several parts:
1.   Loading and Exploration of the Dataset
2.   Creating the vocabulary
3.   Preparing the Dataset
4.   Defining the Architecture of the LSTM with embeddings
5.   Training of the Model

## Exercise 3

### 1. Loading and Exploration of the Dataset

**[1.1]** Install the following packages

In [ ]:
!pip uninstall -y torch torchtext torchvision
!pip install torch==2.3.0 torchtext==0.18.0 torchvision==0.18.0 lightning

**[1.2]** Now we need to import the necessary libraries of PyTorch and Python

In [ ]:
# Solutions
import torch
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
import spacy
from torchtext.vocab import build_vocab_from_iterator
from itertools import chain
from torchtext.data.functional import to_map_style_dataset
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import Vocab
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from sklearn.model_selection import train_test_split
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns
import time

**[1.3]** Enable Synchronous CUDA Operations

Description: Set CUDA_LAUNCH_BLOCKING to '1' to make all CUDA operations synchronous.

Purpose: This setting helps in debugging by forcing CUDA to perform operations synchronously, which provides a more precise error traceback.

In [ ]:
# Solutions
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

**[1.4]** Now we will choose the device type [CPU/GPU] in such a way that if GPU is available then use GPU otherwise use CPU.

In [ ]:
# Solutions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

**[1.5]** Now we will get the data directly from kaggle. To do that we need to upload `kaggle.json` file, downloaded from kaggle while being signed in by using **own account**.

How to get The Kaggle.json:

1. Sign in to your kaggle account and go to the following link `https://www.kaggle.com/settings/account`
2. Scroll down to API
3. Create new token
4. It will download a kaggle.json for you
5. upload the kaggle.json after you run the following cell.

Finally upload the `kaggle.json`

In [ ]:
# Solutions
files.upload()

**[1.6]** Let's connect Kaggle to Google Colab

In [ ]:
# Solutions
!mkdir -p ~/.kaggle # Creates a .kaggle directory, if it does not exist. -p flag is there to make sure there is no error if the directory already exists
!cp kaggle.json ~/.kaggle/ # Copies the kaggle.json, which contains the API credentials to the .kaggle directory. this file is used to get API access.
!chmod 600 ~/.kaggle/kaggle.json  # This changes the permission to read/write for the file. This ensures security measures to protect API credentials.

**[1.7]** Now download the dataset from kaggle: https://www.kaggle.com/datasets/adarshpathak/shakespeare-text


In [ ]:
# Solutions
!kaggle datasets download -d adarshpathak/shakespeare-text

**[1.8]** Unzip the downloaded dataset

In [ ]:
# Solutions
!unzip /content/shakespeare-text.zip

**[1.8]** Now we will create a function named `read_data` to read our data. Unlike the previous examples, here we will be dealing with a .txt file, not a .csv file. So, creating a function to better handle and keep the code clean.

In [ ]:
# Solutions
def read_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    return text

**[1.9]** Let's read the data using a `read_csv` function and save it into panda's dataframe named `text_data`

In [ ]:
# Solutions
# Replace 'path_to_file.txt' with the path to your actual text file
text_data = read_data('/content/text.txt')
print(text_data[:200])

**[1.10]** Print the length of the character we have in this dataset

In [ ]:
# Solutions
print(f"Length of text: {len(text_data)} characters")

# 2. Creating Vocabulary

Since we want to generate Shakespear's text, we will be working with character level tokenization, unlike the previous modules where we worked with word-level tokenization. To capture the precise style and nuance of Shakespear's language, the character-level tokenization is preferred.

**[2.1]** Tokenization

Just convert the text_data into a list using the `list` function, that will create the character level tokenization


In [ ]:
# Solutions
# Tokenize the text into characters
tokens = list(text_data)
tokens[:10]

**[2.2]** Build Vocabulary

This step involves creating a vocabulary from the tokenized text data. The vocabulary is essential for the model as it provides a consistent way to convert characters to integer indices. This conversion is crucial for preparing the input data for training the neural network model, where each character needs to be represented as a numerical input.


In [ ]:
# Solutions
# Build vocabulary from the list of characters
# Special tokens like '<unk>' for unknown characters and '<pad>' for padding are also added to the vocabulary.
vocab = build_vocab_from_iterator([tokens], specials=['<unk>', '<pad>'])
vocab.set_default_index(vocab['<unk>'])  # Set default index for unknown tokens, which handles any characters not present in the vocabulary.

**[2.3]** Check the size of the vocabulary and print the newly created vocabulary.

In [ ]:
# Solutions
# Check the vocabulary size and display some example mappings
vocab_size = len(vocab)

# Print the entire vocabulary
print("Full vocabulary mapping:")
for character in sorted(vocab.get_stoi()):
    index = vocab[character]
    print(f"'{character}': {index}")

**[2.3]** Convert the character tokens into numeric represenation according to the vocabulary

In [ ]:
# Solutions
# Convert tokens to indices according to the vocabulary
numericalized_tokens = [vocab[token] for token in tokens]
print(f"First 100 numericalized tokens: {numericalized_tokens[:100]}")

# Creating an inverse vocabulary mapping from index to character using torchtext's get_itos()
index_to_char = {i: char for i, char in enumerate(vocab.get_itos())}

# Convert the first 100 numericalized tokens back to characters
characters_from_indices = [index_to_char[index] for index in numericalized_tokens[:100]]
print(f"First 100 corresponding chars: {characters_from_indices}")

# 3.   Preparing the Dataset

**[3.1]** **Create Input and Output Sequences**

This section involves creating input and output sequences for the model training. Each input sequence consists of a series of characters (or tokens) from the text, and the corresponding output sequence is the input sequence shifted by one character. This setup is typical for training language models where the aim is to predict the next character in a sequence.

Task: Create a function called "create_input_sequence". The parameter of that function is input data and sequence length.


In [ ]:
# Solutions
def create_inout_sequences(input_data, seq_length):
    """
    Creates pairs of input sequences and the corresponding target sequences,
    where each target is the input sequence shifted by one character.
    Args:
        input_data (list): List of numericalized tokens.
        seq_length (int): The length of each sequence.
    Returns:
        list of tuples: Each tuple contains a pair of torch tensors (input_seq, target_seq).
    """
    sequences = []
    for i in range(len(input_data) - seq_length):
        sequence = input_data[i:i+seq_length]
        target_sequence = input_data[i+1:i+seq_length+1]
        sequences.append((torch.tensor(sequence, dtype=torch.long), torch.tensor(target_sequence, dtype=torch.long)))
    return sequences

**[3.2]** The sequence length is 100 here.

*context-based intuition*

1. The intuition is that the model can use the context of the previous 99 characters to predict the 100th character. It provides context to the model.

2. In many natural language processing tasks, having a longer context can help the model capture more complex patterns, like phrases or clauses in sentences, which can span several characters.

*training-dynamics*
1. The sequence length directly affects how far back in time gradients are propagated while training(backpropagation through time). Longer sequences can add on more comprehensive gradient information.

It can also depend on so many other factors such as performance vs computational cost, model selection, and of course the problem definition.

Task: Display the input features and targets for the first sequence.

In [ ]:
# Solutions
# creating sequences with a specific sequence length
seq_length = 100  # Define the sequence length
sequences = create_inout_sequences(numericalized_tokens, seq_length)

# Display the first sequence and its corresponding target for verification
print("Sample sequence (input):", sequences[0][0])
print("Sample target (shifted by one character):", sequences[0][1])

**[3.3]** **Create Dataset and DataLoader**

This step involves wrapping the created sequences into a custom Dataset class, which is then used to create a DataLoader. The DataLoader facilitates efficient batching, shuffling, and loading of the data during model training. This setup is crucial for handling large datasets and training deep learning models efficiently.

While working with Pytorch, creating several different custom classes is standard practice. This makes the the code clean and reusable if not using in colab.

Task: Create a function called `CharDataset` that will take the sequences and convert it to torch.


In [ ]:
# Solutions
class CharDataset(Dataset):
    """
    A custom Dataset class for handling character-level sequence data for language modeling.

    This dataset class is designed to provide pairs of sequences where each input sequence
    has a corresponding target sequence, which is the input sequence shifted by one character.
    It is typically used for training models to predict the next character in a sequence,
    a common setup for character-level language modeling.

    Attributes:
        sequences (list of tuples): A list where each tuple contains two elements:
            - input_seq (list of int): The input sequence represented as a list of integer indices.
            - target_seq (list of int): The target sequence represented as a list of integer indices.

    Methods:
        __init__(self, sequences): Initializes the dataset with the provided list of sequences.
        __len__(self): Returns the number of sequences in the dataset.
        __getitem__(self, index): Retrieves the input and target sequence pair at the specified index.
    """

    def __init__(self, sequences):
        """
        Initializes the CharDataset instance with a list of sequences.

        Args:
            sequences (list of tuples): Each tuple contains two lists of integers representing
                the input sequence and the target sequence respectively.
        """
        self.sequences = sequences

    def __len__(self):
        """
        Returns the total number of sequences in the dataset.

        Returns:
            int: The number of sequences.
        """
        return len(self.sequences)

    def __getitem__(self, index):
        """
        Retrieves the input and target sequence pair based on the dataset index.

        Args:
            index (int): The index of the sequence pair to retrieve.

        Returns:
            tuple: Contains two torch.Tensors, both of long type:
                - The first tensor is the input sequence.
                - The second tensor is the target sequence (input sequence shifted by one character).
        """
        input_seq, target_seq = self.sequences[index]
        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)

**[3.4]** Create the `DataLoader` for training and save it to train_loader. Set the bacth size as 128.

In [ ]:
# Solutions
# Define the batch size
batch_size = 128

# Create the DataLoader for training
train_loader = DataLoader(CharDataset(sequences), batch_size=batch_size, shuffle=True, drop_last=True)

# Check the DataLoader output to ensure correct batching
for input_example, target_example in train_loader:
    print(f'Input batch shape: {input_example.shape} \nTarget batch shape: {target_example.shape}')
    break

# 4.   Defining the Architecture of the LSTM with embeddings

**[4.1]** Let's create a character-level RNN using LSTM model. This LSTM model consists of 1. an embedding layer, 2. a LSTM layer, 3. a fully connected layer for output predictions.

In [ ]:
# Solutions
import torch
import pytorch_lightning as pl
from torch import nn

class CharRNN(pl.LightningModule):
    """
    A character-level RNN using LSTM for sequence generation.

    Attributes:
        embed (torch.nn.Embedding): Embedding layer to transform indices into dense vectors.
        lstm (torch.nn.LSTM): LSTM layer to handle the sequence input.
        fc (torch.nn.Linear): A fully connected layer to output the probability distribution over the vocabulary.
        criterion (torch.nn.Module): Loss function, CrossEntropyLoss, used for training.
    """
    def __init__(self, vocab_size, embed_dim, rnn_neurons):
        """
        Constructor: Initializes the CharRNN class.

        Args:
            vocab_size (int): Size of the vocabulary.
            embed_dim (int): Dimensionality of the embedding layer.
            rnn_neurons (int): Number of neurons (or units) in the LSTM layer.
        """
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, rnn_neurons, batch_first=True) # batch_first = True - The LSTM expects the input tensor to have a shape of (batch_size, sequence_length, features)
                                                                      # batch_size = the number of sequences in each batch - 128(provided)
                                                                      # sequence_length =  the length of each sequence - 100 (explanation in creating vocab part)
                                                                      # features  = the number of features per time step - embed_dim in this case
        self.fc = nn.Linear(rnn_neurons, vocab_size)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x, hidden):
        """
        Defines the forward pass of the module.

        Args:
            x (torch.Tensor): Input tensor containing batches of input sequences.
            hidden (tuple): The initial hidden and cell state of the LSTM.

        Returns:
            tuple: The output from the Linear layer and the hidden states from the LSTM layer.
        """
        embedding = self.embed(x)
        output, hidden = self.lstm(embedding, hidden)
        output = self.fc(output.contiguous().view(-1, output.shape[-1]))
        return output, hidden

    def training_step(self, batch, batch_idx):
        """
        Performs a single training step, calculates loss, and logs the training loss.

        Args:
            batch (tuple): Contains the input and target sequences.
            batch_idx (int): Index of the batch.

        Returns:
            torch.Tensor: The loss of the training step.
        """
        inputs, targets = batch
        hidden = self.init_hidden(inputs.size(0))
        output, hidden = self(inputs, hidden)
        targets_flattened = targets.view(-1)
        loss = self.criterion(output, targets_flattened)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def configure_optimizers(self):
        """
        Sets up the optimizer for training.

        Returns:
            torch.optim.Optimizer: The optimizer for the model.
        """
        optimizer = torch.optim.Adam(self.parameters())
        return optimizer

    def init_hidden(self, batch_size):
        """
        Initializes the hidden state of the LSTM. Used during Training process and Inference.

        Args:
            batch_size (int): The size of the batch for which to initialize the state.

        Returns:
            tuple: Initial hidden state and cell state for the LSTM.
        """
        return (torch.zeros(1, batch_size, self.lstm.hidden_size, device=self.device),
                torch.zeros(1, batch_size, self.lstm.hidden_size, device=self.device))

**[4.2]** Let's set the model parameters. Set the embedding dimension as 64, rnn_neurons as 1026, batch_size as 128, set the number of epochs as 3.

In [ ]:
# Solutions
# Model parameters
vocab_size = len(vocab)
embed_dim = 64
rnn_neurons = 1026
batch_size = 128
num_epochs = 3

**[4.3]** Let's instantiate the newly created LSTM model and save it to `model`. Finally send the model to available device (CPU/GPU)

In [ ]:
# Solutions
# Instantiate the defined model with parameters
model = CharRNN(vocab_size, embed_dim, rnn_neurons)
model.to(device)  # Move model to GPU if available

# 5.   Training of the Model

**[5.1]** Set up training utilities including importing `Trainer`, `TensorBoardLogger` and callback functions such as `ModelCheckpoint` and `EarlyStopping`.


In [ ]:
# Solutions
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping


# Configure TensorBoard logging for visualizing training progress.
# 'tb_logs' is the directory where TensorBoard logs will be saved.
# 'shakespear's text' is a subdirectory specifically for this training run.
logger = TensorBoardLogger("tb_logs", name="shakespear's text")

**[5.2]** **Configure Model Checkpointing**

This section sets up the model checkpointing mechanism to automatically save the model during training when it achieves better performance based on the monitored metric. This ensures that the best version of the model is retained, which can be crucial for long training processes or when fine-tuning a model where incremental improvements might be significant. Let's introduce a function called `checkpoint_callback`.


In [ ]:
# Solutions
# Set up the model checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best-checkpoint-{epoch:02d}-{train_loss:.2f}',
    save_top_k=1,  # Save only the top 1 checkpoint (best)
    verbose=True,
    monitor='train_loss',  # Ideally we replace 'train_loss' with 'val_loss' if we have validation dataset, which is not the case for this model.
    mode='min',
    auto_insert_metric_name=False  # Avoids appending the metric name to the filename if a cleaner filename is preferred.
)

**[5.3]** **Set Up Early Stopping**

Now we will introduce a function called `early_stopping_callback`.This section configures early stopping to prevent overfitting and reduce computational waste. Early stopping will halt the training process if the monitored metric does not improve for a set number of epochs, which is defined by the `patience` parameter.


In [ ]:
# Solutions
# Set up the early stopping callback to halt training if 'train_loss' does not improve.
early_stopping_callback = EarlyStopping(
    monitor='train_loss',  # Metric to monitor for improvement.
    patience=2,  # Number of epochs with no improvement after which training will be stopped.
    strict=False,  # If True, will raise an error if monitoring metric is not available.
    verbose=True,  # Enable verbosity to print logs when training is stopped.
    mode='min'  # 'min' indicates that a decrease in 'train_loss' is considered an improvement.
)

**[5.4]** Configure the Trainer

This section initializes the PyTorch Lightning `Trainer`(https://lightning.ai/docs/pytorch/stable/common/trainer.html), which orchestrates the model training. It is configured with the previously set up logging, checkpointing, and early stopping mechanisms. The trainer manages the training process, leveraging GPU acceleration if available, and logs every 10 steps.


In [ ]:
# Solutions
# Initialize the PyTorch Lightning Trainer with the specified configurations.
trainer = Trainer(
    max_epochs=num_epochs,  # Maximum number of epochs to train.
    logger=logger,  # Configure logging to TensorBoard.
    callbacks=[checkpoint_callback, early_stopping_callback],  # List of callbacks for checkpointing and early stopping.
    log_every_n_steps=10  # Frequency of logging within an epoch.
)

**[5.5]** Train the model by calling `trainer.fit` function of lightning (https://lightning.ai/docs/pytorch/stable/common/trainer.html)

In [ ]:
# Solutions
# Start training
trainer.fit(model, train_loader)

**[5.6]** Save the last checkpoint after training

In [ ]:
# Solutions
# Optional: Save the last checkpoint explicitly if needed
trainer.save_checkpoint("final_model.ckpt")

**[5.7]** Load the tensorboard to monitor performance

In [ ]:
# Solutions
%load_ext tensorboard
%tensorboard --logdir tb_logs

**[5.8]** Load the saved model

In [ ]:
# Solutions
# Assuming the path to the checkpoint, vocabulary size, embedding dimension, and RNN neuron count
checkpoint_path = '/content/checkpoints/best-checkpoint-02-0.24.ckpt'

**[5.8]** Use the function you defined earlier to create a second model

In [ ]:
# Solutions
model2 = CharRNN.load_from_checkpoint(
        checkpoint_path=checkpoint_path,
        vocab_size=vocab_size,
        embed_dim=embed_dim,
        rnn_neurons=rnn_neurons
    )
model2.to(device)

 **[5.9]** **Create Vocabulary Mappings**

This section describes the setup of two essential vocabulary mappings: `itos` (Index-to-Token) and `stoi` (Token-to-Index). These mappings are fundamental for encoding and decoding processes in natural language processing tasks. They allow the model to convert back and forth between human-readable text (tokens) and machine-readable numerical formats (indices), which is critical for both training and generating predictions.


`itos (Index-to-Token)`: This dictionary maps numerical indices to their corresponding tokens (such as characters or words). It is primarily used during the decoding process, where the model's output, typically numerical, needs to be converted back into human-readable text. For example, after predicting sequences of indices in text generation, itos helps translate these indices into actual characters or words.

`stoi (Token-to-Index)`: Conversely, this dictionary maps tokens to numerical indices. It is crucial for encoding the text data into a format that the model can understand and process. Before feeding textual data into the model for training or inference, stoi is used to convert characters or words into a sequence of indices.

Tasks: Define two functions that works as described above and save it to 1. itos and 2. stoi

In [ ]:
# Solutions
itos = {i: tok for i, tok in enumerate(vocab.get_itos())}
stoi = {tok: i for i, tok in enumerate(vocab.get_itos())}

**[5.10]** Generate Text Using the Trained Model

This section defines and utilizes a function, `generate_text`, to generate text based on a given starting string using a trained LSTM model. The function processes the input string to generate a specified number of characters, building on the initial input by predicting one character at a time. This demonstrates the model's ability to use learned patterns to create plausible text sequences, which is essential for applications like chatbots, automated writing assistants, or other generative tasks.

Task: Create a function that will be called to generate text


In [ ]:
# Solutions
def generate_text(model, start_str, gen_size=100, device=device):
    """
    Generates text from a trained model based on an input starting string.

    Args:
        model (torch.nn.Module): The trained model to use for text generation.
        start_str (str): The initial string to start text generation.
        gen_size (int): The number of characters to generate.
        device (torch.device): The device (CPU/GPU) on which to perform the computation.

    Returns:
        str: The original start_str extended by gen_size characters generated by the model.
    """
    model.eval()  # Switch the model to evaluation mode (i.e., turn off dropout, etc.). Standard practice for inference after loading the model.

    # Convert the start_str to a tensor of indices suitable for the model input.
    input_seq = torch.tensor([stoi[char] for char in start_str if char in stoi], dtype=torch.long, device=device).unsqueeze(0)
    text_generated = []  # List to collect the generated characters.
    hidden = model.init_hidden(1)  # Initialize the hidden state of the LSTM.

    # Generate text without calculating gradients to speed up the process and reduce memory usage.
    with torch.no_grad():
        for _ in range(gen_size):
            output, hidden = model(input_seq, hidden)  # Forward pass through the model.
            predicted_id = torch.argmax(output[-1], dim=0)  # Get the most likely next character index.
            predicted_char = itos[predicted_id.item()]  # Convert the index back to a character.
            text_generated.append(predicted_char)  # Append the predicted character to the output list.
            input_seq = predicted_id.unsqueeze(0).unsqueeze(0)  # Update the input for the next prediction.

    return start_str + ''.join(text_generated)  # Return the original string extended by the generated text.

**[5.11]** Generate 300 characters with dagger as the starting sequence

In [ ]:
# Solutions
# Ensure you call this on the correct model instance
print(generate_text(model2, "dagger, ", 300))